# Task 2: Exploratory Data Analysis (EDA) with Pandas

**Internship:** CodeOrbit Tech — Data Science Internship  
**Dataset:** `cleaned_sales_data.csv`, prepared in Task 1 (fictional practice data)

## Objectives
- Summarize the dataset using Pandas.
- Explore sales distributions, product/category/region differences, and daily changes.
- Create histograms, bar charts, a line chart, and a scatter plot using Matplotlib and Seaborn.
- Identify possible outliers and write evidence-based findings.

**How to use:** Keep `cleaned_sales_data.csv` in the same folder as this notebook. Run cells from top to bottom. In Google Colab, upload the CSV using the folder icon on the left if it is not already present.

> The dataset is small and fictional. It is suitable for learning EDA, not for drawing conclusions about a real business.

## 1. Import libraries and load the data

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

Path("eda_charts").mkdir(exist_ok=True)
INPUT_FILE = Path("cleaned_sales_data.csv")
if not INPUT_FILE.exists():
    raise FileNotFoundError("Upload cleaned_sales_data.csv and keep it beside this notebook.")

df = pd.read_csv(INPUT_FILE)
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Date range: {df['Order_Date'].min():%d %b %Y} to {df['Order_Date'].max():%d %b %Y}")
display(df.head())

## 2. Inspect data quality and column types

In [ ]:
print("Column data types:")
print(df.dtypes)

print("\nMissing values per column:")
display(df.isna().sum().to_frame("Missing values"))

print("\nDuplicate rows:", int(df.duplicated().sum()))

## 3. Descriptive statistics

In [ ]:
numeric_columns = ["Quantity", "Unit_Price", "Customer_Age", "Sales_Amount"]
display(df[numeric_columns].describe().round(2))

print(f"Total sales: ₹{df['Sales_Amount'].sum():,.2f}")
print(f"Mean sales per order: ₹{df['Sales_Amount'].mean():,.2f}")
print(f"Median sales per order: ₹{df['Sales_Amount'].median():,.2f}")

## 4. Category counts and grouped summaries

In [ ]:
for column in ["Product", "Category", "Region", "Payment_Method"]:
    print(f"\nCounts for {column}:")
    display(df[column].value_counts().rename_axis(column).to_frame("Orders"))

product_summary = df.groupby("Product").agg(
    Orders=("Order_ID", "count"),
    Units_Sold=("Quantity", "sum"),
    Total_Sales=("Sales_Amount", "sum"),
    Average_Order_Sales=("Sales_Amount", "mean")
).sort_values("Total_Sales", ascending=False).round(2)

display(product_summary)

category_summary = df.groupby("Category").agg(
    Orders=("Order_ID", "count"), Units_Sold=("Quantity", "sum"),
    Total_Sales=("Sales_Amount", "sum")
).sort_values("Total_Sales", ascending=False)
display(category_summary)

region_summary = df.groupby("Region").agg(
    Orders=("Order_ID", "count"), Total_Sales=("Sales_Amount", "sum"),
    Average_Order_Sales=("Sales_Amount", "mean")
).sort_values("Total_Sales", ascending=False).round(2)
display(region_summary)

## 5. Histogram: distribution of sales amounts

In [ ]:
sns.set_theme(style="whitegrid", context="notebook")
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(data=df, x="Sales_Amount", bins=8, kde=True, ax=ax, color="#167D9A")
ax.axvline(df["Sales_Amount"].mean(), color="#C44E52", linestyle="--", label=f"Mean = ₹{df['Sales_Amount'].mean():,.0f}")
ax.axvline(df["Sales_Amount"].median(), color="#4C72B0", linestyle=":", label=f"Median = ₹{df['Sales_Amount'].median():,.0f}")
ax.set(title="Distribution of Sales Amount per Order", xlabel="Sales amount (₹)", ylabel="Number of orders")
ax.legend()
fig.tight_layout()
fig.savefig("eda_charts/01_sales_amount_distribution.png", dpi=180, bbox_inches="tight")
plt.show()

**Interpretation:** The mean (about ₹518) is higher than the median (₹385). A few larger transactions pull the mean upward, so the typical order is better represented by the median in this small sample.

## 6. Bar chart: order count by product

In [ ]:
product_orders = df["Product"].value_counts().sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 5))
product_orders.plot(kind="barh", ax=ax, color="#4C9F70")
ax.set(title="Number of Orders by Product", xlabel="Number of orders", ylabel="Product")
for i, val in enumerate(product_orders.values):
    ax.text(val + 0.08, i, str(val), va="center", fontsize=9)
ax.set_xlim(0, max(product_orders.values) + 1.5)
fig.tight_layout()
fig.savefig("eda_charts/02_orders_by_product.png", dpi=180, bbox_inches="tight")
plt.show()

**Interpretation:** Notebook is the most frequently ordered product (8 orders). Order count measures popularity by number of orders; it does not measure total sales value.

## 7. Bar chart: total sales by product and category

In [ ]:
product_sales = df.groupby("Product")["Sales_Amount"].sum().sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 5))
product_sales.plot(kind="barh", ax=ax, color="#D18F36")
ax.set(title="Total Sales by Product", xlabel="Total sales (₹)", ylabel="Product")
for i, val in enumerate(product_sales.values):
    ax.text(val + 45, i, f"₹{val:,.0f}", va="center", fontsize=9)
ax.set_xlim(0, max(product_sales.values) * 1.18)
fig.tight_layout()
fig.savefig("eda_charts/03_revenue_by_product.png", dpi=180, bbox_inches="tight")
plt.show()

category_sales = df.groupby("Category")["Sales_Amount"].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
category_sales.plot(kind="bar", ax=ax, color=["#4C9F70", "#D18F36", "#8172B2"])
ax.set(title="Total Sales by Category", xlabel="Category", ylabel="Total sales (₹)")
ax.tick_params(axis="x", rotation=15)
for i, val in enumerate(category_sales.values):
    ax.text(i, val + 100, f"₹{val:,.0f}", ha="center", fontsize=9)
ax.set_ylim(0, max(category_sales.values) * 1.18)
fig.tight_layout()
fig.savefig("eda_charts/07_sales_by_category.png", dpi=180, bbox_inches="tight")
plt.show()

**Interpretation:** Backpack contributes the highest total sales (₹6,293). Accessories is the largest category by sales (₹9,543), while Stationery has the most orders and units but a much lower sales total because its items have lower prices.

## 8. Bar chart: sales by region

In [ ]:
region_sales = df.groupby("Region")["Sales_Amount"].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=region_sales.values, y=region_sales.index, ax=ax, color="#8172B2")
ax.set(title="Total Sales by Region", xlabel="Total sales (₹)", ylabel="Region")
for i, val in enumerate(region_sales.values):
    ax.text(val + 35, i, f"₹{val:,.0f}", va="center", fontsize=9)
ax.set_xlim(0, max(region_sales.values) * 1.18)
fig.tight_layout()
fig.savefig("eda_charts/04_sales_by_region.png", dpi=180, bbox_inches="tight")
plt.show()

**Interpretation:** Chennai has the most orders (10) and the highest total sales (₹4,917). Hyderabad has the highest average sales per order (about ₹643.86), illustrating that order count and average order value tell different stories.

## 9. Line chart: daily sales trend

In [ ]:
daily_sales = df.groupby("Order_Date", as_index=False)["Sales_Amount"].sum().sort_values("Order_Date")
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=daily_sales, x="Order_Date", y="Sales_Amount", marker="o", ax=ax, color="#167D9A")
ax.set(title="Daily Sales Trend", xlabel="Order date", ylabel="Total sales per day (₹)")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig("eda_charts/05_daily_sales_trend.png", dpi=180, bbox_inches="tight")
plt.show()

best_day = daily_sales.loc[daily_sales["Sales_Amount"].idxmax()]
print(f"Highest-sales day: {best_day['Order_Date']:%d %b %Y} — ₹{best_day['Sales_Amount']:,.0f}")

**Interpretation:** Sales fluctuate from day to day rather than following a steady trend. The highest daily total is ₹2,548 on 8 January 2026. Because the sample spans only 23 days and contains 30 orders, this should not be treated as a reliable seasonal pattern.

## 10. Scatter plot: unit price vs sales amount

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df, x="Unit_Price", y="Sales_Amount", hue="Category", s=90, ax=ax, palette="Set2")
ax.set(title="Unit Price vs. Sales Amount", xlabel="Unit price (₹)", ylabel="Sales amount per order (₹)")
ax.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
fig.savefig("eda_charts/06_unit_price_vs_sales.png", dpi=180, bbox_inches="tight")
plt.show()

print(f"Correlation, unit price vs sales amount: {df['Unit_Price'].corr(df['Sales_Amount']):.3f}")
print(f"Correlation, quantity vs sales amount: {df['Quantity'].corr(df['Sales_Amount']):.3f}")

**Interpretation:** Unit price and sales amount show a strong positive correlation (about 0.861). This is expected because sales amount is calculated from quantity × unit price. Quantity has a negative correlation with sales in this sample, but this does **not** mean selling more units reduces sales; lower-priced products tend to have larger quantities.

## 11. Check possible outliers

In [ ]:
q1 = df["Sales_Amount"].quantile(0.25)
q3 = df["Sales_Amount"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
possible_outliers = df.loc[df["Sales_Amount"] > upper_bound, ["Order_ID", "Product", "Quantity", "Unit_Price", "Sales_Amount"]]
print(f"IQR upper bound: ₹{upper_bound:,.2f}")
print(f"Potential high-value orders: {len(possible_outliers)}")
display(possible_outliers)

**Interpretation:** Two Backpack orders of ₹1,798 exceed the IQR threshold. They are plausible high-value orders (2 units × ₹899), so they should be reviewed rather than automatically removed.

## 12. Final findings and limitations

1. **Overall:** 30 cleaned orders generated **₹15,538** in total sales. Mean sales per order were **₹517.93**, while median sales were **₹385.00**.
2. **Product:** Backpack generated the highest product sales (**₹6,293**), while Notebook had the most orders (**8**).
3. **Category:** Accessories led sales (**₹9,543**); Stationery had the highest order count (**14**) but only ₹2,095 in sales.
4. **Region:** Chennai had the most orders (**10**) and the highest regional sales (**₹4,917**). Hyderabad had the highest average sales per order (about ₹643.86).
5. **Payment:** UPI was used for **15 of 30 orders**.
6. **Time and outliers:** 8 January 2026 had the highest daily sales (**₹2,548**). Two ₹1,798 Backpack orders were flagged by the IQR rule but appear plausible.

**Limitations:** The dataset is a small fictional sample. The results are useful for demonstrating descriptive statistics, grouped summaries, visualizations, correlations, and outlier checks; they should not be generalized to real customers or used as business forecasts.

## Optional: export analysis tables

In [ ]:
product_summary.to_csv("product_summary.csv")
category_summary.to_csv("category_summary.csv")
region_summary.to_csv("region_summary.csv")
payment_summary = df.groupby("Payment_Method").agg(
    Orders=("Order_ID", "count"), Total_Sales=("Sales_Amount", "sum")
).sort_values("Orders", ascending=False)
payment_summary.to_csv("payment_summary.csv")
daily_sales.to_csv("daily_sales_summary.csv", index=False)
print("Saved product_summary.csv, category_summary.csv, region_summary.csv, payment_summary.csv, and daily_sales_summary.csv")